# Recurrent Neural Networks and Sequence Models

Images have spatial structure — CNNs exploit it. **Sequences** have temporal structure: words in a sentence, notes in a melody, readings from a sensor over time. The meaning of each element depends on what came before (and often what comes after). Feedforward networks process fixed-size inputs independently. **Recurrent Neural Networks (RNNs)** maintain memory across time steps, making them natural models for sequential data.

This notebook covers how RNNs work, their evolution through LSTM and GRU architectures, sequence modeling patterns, and the applications — from language modeling to time series forecasting — that RNNs enabled before the Transformer era.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. Sequential Data

Sequential data has an inherent order. Examples:

| Domain | Sequence | Element |
|--------|----------|--------|
| Language | Sentence | Word or character |
| Speech | Audio waveform | Amplitude sample |
| Finance | Stock prices | Daily closing price |
| Healthcare | Patient records | Vital sign reading |
| Music | Composition | Note |

The key challenge: the input length varies (sentences have different numbers of words), and context matters. The word "bank" means something different after "river" than after "investment."

**Sequence modeling tasks:**

- **Many-to-one** — entire sequence → single output (sentiment analysis, video classification).
- **One-to-many** — single input → sequence output (image captioning).
- **Many-to-many (aligned)** — sequence → sequence of same length (part-of-speech tagging).
- **Many-to-many (unaligned)** — sequence → different length sequence (machine translation).

---

## 2. The Vanilla RNN

A recurrent neural network adds a **hidden state** $\mathbf{h}_t$ that carries information from previous time steps. At each step $t$, the network receives input $\mathbf{x}_t$ and the previous hidden state $\mathbf{h}_{t-1}$, and produces output $\mathbf{y}_t$ and updated hidden state $\mathbf{h}_t$.

$$\mathbf{h}_t = \tanh(\mathbf{W}_{hh} \mathbf{h}_{t-1} + \mathbf{W}_{xh} \mathbf{x}_t + \mathbf{b}_h)$$

$$\mathbf{y}_t = \mathbf{W}_{hy} \mathbf{h}_t + \mathbf{b}_y$$

The same weights $\mathbf{W}_{hh}$, $\mathbf{W}_{xh}$, $\mathbf{W}_{hy}$ are applied at every time step — **weight sharing across time**. This is analogous to how CNNs share weights across space.

The hidden state $\mathbf{h}_t$ is the network's **memory** — a compressed representation of everything it has seen up to time $t$.

---

## 3. Unfolding Through Time

An RNN can be visualized as a feedforward network **unfolded** across time steps:

```
  x₁        x₂        x₃        x₄
  ↓         ↓         ↓         ↓
  RNN  →h₁→ RNN  →h₂→ RNN  →h₃→ RNN  →h₄
  ↓         ↓         ↓         ↓
  y₁        y₂        y₃        y₄
```

Each "RNN block" uses the same parameters. Backpropagation through this unfolded graph is called **Backpropagation Through Time (BPTT)** — the same chain rule from standard backprop, applied across time steps.

BPTT can be **truncated** — backpropagate only through the last $K$ time steps rather than the entire sequence — to reduce computation and memory for very long sequences.

In [ ]:
class VanillaRNN:
    """Simple RNN with tanh activation."""

    def __init__(self, input_size, hidden_size, output_size):
        scale = 0.01
        self.W_xh = np.random.randn(hidden_size, input_size) * scale
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale
        self.W_hy = np.random.randn(output_size, hidden_size) * scale
        self.b_h = np.zeros((hidden_size, 1))
        self.b_y = np.zeros((output_size, 1))
        self.hidden_size = hidden_size

    def forward(self, inputs):
        """inputs: list of input vectors, one per time step."""
        h = np.zeros((self.hidden_size, 1))
        outputs, hidden_states = [], []

        for x in inputs:
            x = x.reshape(-1, 1)
            h = np.tanh(self.W_xh @ x + self.W_hh @ h + self.b_h)
            y = self.W_hy @ h + self.b_y
            hidden_states.append(h.copy())
            outputs.append(y)

        return outputs, hidden_states


# Forward pass on a short sequence
rnn = VanillaRNN(input_size=3, hidden_size=4, output_size=2)
sequence = [np.array([1.0, 0.0, 0.0]),
            np.array([0.0, 1.0, 0.0]),
            np.array([0.0, 0.0, 1.0]),
            np.array([1.0, 1.0, 0.0])]

outputs, hiddens = rnn.forward(sequence)

print(f"Sequence length: {len(sequence)}")
print(f"Hidden size: {rnn.hidden_size}")
for t, (y, h) in enumerate(zip(outputs, hiddens)):
    print(f"  t={t}: h={h.flatten().round(3)}, y={y.flatten().round(3)}")

---

## 4. The Vanishing Gradient Problem

During BPTT, gradients flow backward through time, multiplied by $\mathbf{W}_{hh}$ at each step. Over $T$ time steps:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{h}_1} \propto \left(\mathbf{W}_{hh}\right)^{T-1}$$

If the largest eigenvalue of $\mathbf{W}_{hh}$ is less than 1, gradients **vanish** exponentially — early time steps receive almost no learning signal. The network cannot learn **long-range dependencies**.

If the eigenvalue exceeds 1, gradients **explode** — training becomes unstable.

Tanh and sigmoid activations make this worse — their derivatives are ≤ 1, further shrinking gradients at each step. This limits vanilla RNNs to roughly 10–20 time steps of effective memory.

In [ ]:
# Gradient decay over time steps
T_steps = 50
W_hh_small = 0.5   # eigenvalue < 1 → vanish
W_hh_large = 1.5   # eigenvalue > 1 → explode
tanh_deriv_max = 1.0

vanish = [(W_hh_small * tanh_deriv_max)**t for t in range(T_steps)]
explode = [(W_hh_large * tanh_deriv_max)**t for t in range(T_steps)]

plt.figure(figsize=(9, 4))
plt.plot(vanish, "b-", linewidth=2, label="Vanishing (|λ| < 1)")
plt.plot(explode, "r-", linewidth=2, label="Exploding (|λ| > 1)")
plt.xlabel("Time steps back")
plt.ylabel("Gradient magnitude")
plt.title("RNN gradients over time: vanish or explode")
plt.yscale("log")
plt.legend()
plt.show()

---

## 5. LSTM — Long Short-Term Memory

LSTM (Hochreiter & Schmidhuber, 1997) solves the vanishing gradient problem with a **cell state** $\mathbf{c}_t$ — a highway for information to flow across many time steps with minimal degradation.

LSTM uses **gates** — sigmoid layers that output values between 0 and 1, controlling what information passes through:

**Forget gate** — what to discard from cell state:

$$\mathbf{f}_t = \sigma(\mathbf{W}_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f)$$

**Input gate** — what new information to store:

$$\mathbf{i}_t = \sigma(\mathbf{W}_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i)$$

$$\tilde{\mathbf{c}}_t = \tanh(\mathbf{W}_c [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c)$$

**Cell state update:**

$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$$

**Output gate** — what to output from cell state:

$$\mathbf{o}_t = \sigma(\mathbf{W}_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o)$$

$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$$

The cell state $\mathbf{c}_t$ can flow unchanged (when $\mathbf{f}_t \approx 1$) across hundreds of time steps — solving long-range dependency learning.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

class LSTMCell:
    """Single LSTM cell forward pass."""

    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        concat_size = input_size + hidden_size
        scale = 0.01
        self.W_f = np.random.randn(hidden_size, concat_size) * scale
        self.W_i = np.random.randn(hidden_size, concat_size) * scale
        self.W_c = np.random.randn(hidden_size, concat_size) * scale
        self.W_o = np.random.randn(hidden_size, concat_size) * scale
        self.b_f = np.zeros((hidden_size, 1))
        self.b_i = np.zeros((hidden_size, 1))
        self.b_c = np.zeros((hidden_size, 1))
        self.b_o = np.zeros((hidden_size, 1))

    def forward(self, x, h_prev, c_prev):
        x = x.reshape(-1, 1)
        concat = np.vstack([h_prev, x])

        f = sigmoid(self.W_f @ concat + self.b_f)   # forget gate
        i = sigmoid(self.W_i @ concat + self.b_i)   # input gate
        c_tilde = np.tanh(self.W_c @ concat + self.b_c)  # candidate
        o = sigmoid(self.W_o @ concat + self.b_o)   # output gate

        c = f * c_prev + i * c_tilde
        h = o * np.tanh(c)

        return h, c, {"f": f, "i": i, "o": o, "c_tilde": c_tilde}


# LSTM forward on a sequence
lstm = LSTMCell(input_size=2, hidden_size=3)
h = np.zeros((3, 1))
c = np.zeros((3, 1))

inputs = [np.array([1.0, 0.5]), np.array([0.0, 1.0]), np.array([0.5, 0.5])]

print("LSTM forward pass:")
for t, x in enumerate(inputs):
    h, c, gates = lstm.forward(x, h, c)
    print(f"  t={t}: forget={gates['f'].flatten().round(2)}, "
          f"input={gates['i'].flatten().round(2)}, "
          f"cell={c.flatten().round(3)}, h={h.flatten().round(3)}")

---

## 6. GRU — Gated Recurrent Unit

GRU (Cho et al., 2014) simplifies LSTM by combining the cell state and hidden state, and using only two gates:

**Reset gate** — how much past to forget:

$$\mathbf{r}_t = \sigma(\mathbf{W}_r [\mathbf{h}_{t-1}, \mathbf{x}_t])$$

**Update gate** — how much of the new state to blend in (analogous to LSTM's forget + input gates):

$$\mathbf{z}_t = \sigma(\mathbf{W}_z [\mathbf{h}_{t-1}, \mathbf{x}_t])$$

$$\tilde{\mathbf{h}}_t = \tanh(\mathbf{W}_h [\mathbf{r}_t \odot \mathbf{h}_{t-1}, \mathbf{x}_t])$$

$$\mathbf{h}_t = (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$$

GRU has fewer parameters than LSTM and often performs comparably. It is a popular choice when computational efficiency matters.

---

## 7. Bidirectional RNNs

Standard RNNs only see the **past** — they process sequences left to right. But context often comes from both directions: understanding a word may require seeing what comes after it.

A **bidirectional RNN** runs two RNNs:
- **Forward RNN** — processes $x_1, x_2, \ldots, x_T$ left to right.
- **Backward RNN** — processes $x_T, x_{T-1}, \ldots, x_1$ right to left.

Their hidden states are concatenated: $\mathbf{h}_t = [\overrightarrow{\mathbf{h}}_t; \overleftarrow{\mathbf{h}}_t]$.

Bidirectional models excel at **sequence labeling** (tagging each element) but cannot be used for **autoregressive generation** (predicting the next token), since the future would not be available at inference time.

---

## 8. Sequence-to-Sequence Models

**Seq2seq** (Sutskever et al., 2014) uses an **encoder-decoder** architecture for tasks like machine translation:

```
Encoder RNN:  "Bonjour" → h₁ → h₂ → h₃ → context vector
Decoder RNN:  context → "Hello" → "world" → <EOS>
```

1. **Encoder** — reads the input sequence and compresses it into a fixed-size **context vector** (final hidden state).
2. **Decoder** — generates the output sequence one token at a time, conditioned on the context vector and its own previous outputs.

**Attention mechanism** (Bahdanau, 2015) improved seq2seq by letting the decoder **attend** to all encoder hidden states, not just the final one. This solved the bottleneck of compressing an entire sentence into one vector and enabled much better translation.

Attention was the conceptual bridge to **Transformers**, which replaced recurrence entirely with self-attention.

---

## 9. Time Series Prediction with RNNs

RNNs naturally model time series — each observation is a time step. A many-to-one RNN reads the entire history and predicts the next value (or a sequence of future values).

In [ ]:
# Simple time series: predict next value using running hidden state
T = 100
t = np.arange(T)
series = np.sin(0.1 * t) + 0.3 * np.sin(0.3 * t) + 0.1 * np.random.randn(T)

# Create input sequences of length 10 → predict next value
seq_len = 10
X_seq, y_seq = [], []
for i in range(len(series) - seq_len):
    X_seq.append(series[i:i+seq_len])
    y_seq.append(series[i+seq_len])

X_seq, y_seq = np.array(X_seq), np.array(y_seq)

# Simple linear predictor as baseline (RNN would learn nonlinear patterns)
from numpy.linalg import lstsq
X_bias = np.hstack([X_seq, np.ones((len(X_seq), 1))])
weights, _, _, _ = lstsq(X_bias, y_seq, rcond=None)
predictions = X_bias @ weights

plt.figure(figsize=(10, 4))
plt.plot(series, "b-", alpha=0.5, label="Original series")
plt.plot(range(seq_len, T), predictions, "r-", linewidth=2, label="Linear predictor")
plt.xlabel("Time")
plt.ylabel("Value")
plt.title("Time series prediction (linear baseline; RNN captures nonlinear patterns)")
plt.legend()
plt.show()

mse = np.mean((predictions - y_seq)**2)
print(f"Linear predictor MSE: {mse:.4f}")

---

## 10. Character-Level Language Modeling

A classic RNN application: train on text, predict the next character. The RNN learns spelling patterns, word structure, and sentence grammar from character sequences.

At each time step:
- **Input:** one-hot encoded character.
- **Output:** probability distribution over all characters.
- **Loss:** cross-entropy between predicted and actual next character.

After training, **generation** works by sampling: feed a seed string, predict the next character, append it, repeat.

In [ ]:
# Character-level modeling: build vocabulary and one-hot encoding
text = "deep learning enables machines to learn from data and make predictions"
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
vocab_size = len(chars)

def one_hot(index, size):
    vec = np.zeros(size)
    vec[index] = 1.0
    return vec

# Convert text to one-hot sequence
sequence = [one_hot(char_to_idx[c], vocab_size) for c in text]

print(f"Text length: {len(text)}")
print(f"Vocabulary ({vocab_size} chars): {''.join(chars)}")
print(f"One-hot vector size: {vocab_size}")
print(f"\nFirst 5 chars -> indices: {[char_to_idx[c] for c in text[:5]]}")

# RNN for character prediction
char_rnn = VanillaRNN(input_size=vocab_size, hidden_size=16, output_size=vocab_size)
outputs, _ = char_rnn.forward(sequence[:8])
print(f"\nRNN output at t=7 (predicting char after '{text[7]}'):")
probs = np.exp(outputs[7].flatten())
probs /= probs.sum()
top3 = np.argsort(probs)[-3:][::-1]
print(f"  Top 3 predictions: {[(idx_to_char[i], f'{probs[i]:.3f}') for i in top3]}")

---

## 11. RNN vs CNN vs Transformer

| Aspect | RNN/LSTM | CNN | Transformer |
|--------|----------|-----|-------------|
| **Data type** | Sequences | Images, spatial | Sequences, any |
| **Processing** | Sequential (step by step) | Parallel (all positions) | Parallel (all positions) |
| **Memory** | Hidden state | Receptive field | Attention over all tokens |
| **Long-range deps** | LSTM/GRU (limited) | Stacked layers | Self-attention (direct) |
| **Training speed** | Slow (sequential) | Fast (parallel) | Fast (parallel) |
| **Dominant era** | 2014–2018 NLP | 2012–present vision | 2018–present NLP/GenAI |

Transformers have largely replaced RNNs for NLP, but RNNs remain relevant for:
- Streaming/online processing (data arrives one step at a time).
- Low-resource environments (fewer parameters than Transformers).
- Time series with clear temporal ordering.
- Audio and speech processing.

---

## 12. Applications of Sequence Models

**Natural Language Processing** — language modeling, machine translation, text generation, named entity recognition, sentiment analysis.

**Speech Recognition** — converting audio waveforms to text. DeepSpeech, Listen-Attend-Spell.

**Time Series Forecasting** — stock prices, weather, energy demand, sensor readings.

**Music Generation** — modeling note sequences. Magenta, WaveNet (which uses dilated convolutions, a CNN variant for sequences).

**Video Analysis** — frame sequences for action recognition, video captioning.

**Healthcare** — patient vital sign monitoring, ECG analysis, disease progression modeling.

---

## 13. Summary

**Recurrent Neural Networks** process sequential data by maintaining a hidden state across time steps. The same weights are applied at each step, with Backpropagation Through Time training the network across the unfolded graph.

Vanilla RNNs suffer from vanishing gradients, limiting long-range learning. **LSTM** solves this with a cell state and gating mechanism (forget, input, output gates). **GRU** provides a simpler gated alternative. **Bidirectional RNNs** capture context from both directions. **Seq2seq** with attention enabled machine translation and led to Transformers.

While Transformers dominate modern NLP, RNNs remain valuable for time series, streaming data, and resource-constrained applications — and their gating concepts (forgetting, selective updating) influenced the design of modern architectures.